# Data Preprocesses and Clean
Hallo! I'm going to combine all the raw order forms into one dataframe for data exploration, removing incomplete orders, converting timestamps to datetime objects, and sorting the dataframe by timestamp.


**Install libraries**

In [42]:
%pip install pandas
%pip install regex

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


**Import libraries for data exploration**

In [43]:
import pandas as pd
import os

I want to combine all the raw order forms into one dataframe for data exploration. First, I'm going to create dataframes for each order form and combine them into one dataframe, creating a unique column for each order form using the first timestamp of each order form.

In [44]:
working_dir = os.getcwd()
order_forms_raw_dir = os.path.join(working_dir, 'order_forms_raw')
order_forms_processed_dir = os.path.join(working_dir, 'order_forms_processed')

df = None

for file in os.listdir(order_forms_raw_dir):
    if file.endswith('.csv'):
        # read the current CSV file
        current_df = pd.read_csv(f'order_forms_raw/{file}')
        
        # get first timestamp from this file to use as identifier
        form_id = current_df['Timestamp'].iloc[0]
        
        # add column identifying which form this came from
        current_df['Form_ID'] = form_id
        
        # add to main dataframe
        if df is None:
            df = current_df
        else:
            df = pd.concat([df, current_df])

if not os.path.exists(order_forms_processed_dir):
    os.makedirs(order_forms_processed_dir)

df.to_csv(os.path.join(order_forms_processed_dir, 'combined_order_forms.csv'), index=False)
print(df.info())


<class 'pandas.core.frame.DataFrame'>
Index: 34060 entries, 0 to 5409
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Timestamp                    14353 non-null  object 
 1   TEA                          14294 non-null  object 
 2   FLAVOR                       5667 non-null   object 
 3   MILK                         9280 non-null   object 
 4   ICE [Ice Level]              10825 non-null  object 
 5   SWEETNESS [Sweetness Level]  11013 non-null  object 
 6   Toppings                     13139 non-null  object 
 7   NAME                         14342 non-null  object 
 8   Unnamed: 8                   0 non-null      float64
 9   Started                      34048 non-null  object 
 10  Completed                    34040 non-null  object 
 11  Unnamed: 11                  8 non-null      object 
 12  Form_ID                      34060 non-null  object 
dtypes: float64(1), object(

Next, I want to remove rows with no timestamps and rows with duplicate timestamps.

In [45]:
df = df[df['Timestamp'].notna()] # remove rows with no timestamps
df = df.drop_duplicates(subset=['Timestamp']) # remove duplicate timestamps

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 10637 entries, 0 to 1122
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Timestamp                    10637 non-null  object 
 1   TEA                          10589 non-null  object 
 2   FLAVOR                       4197 non-null   object 
 3   MILK                         6873 non-null   object 
 4   ICE [Ice Level]              8017 non-null   object 
 5   SWEETNESS [Sweetness Level]  8181 non-null   object 
 6   Toppings                     9702 non-null   object 
 7   NAME                         10628 non-null  object 
 8   Unnamed: 8                   0 non-null      float64
 9   Started                      10629 non-null  object 
 10  Completed                    10623 non-null  object 
 11  Unnamed: 11                  5 non-null      object 
 12  Form_ID                      10637 non-null  object 
dtypes: float64(1), object(

Next, I'm going to remove columns "Unnamed: 8" and "Unnamed: 11"

In [46]:
df = df.drop(columns=["Unnamed: 8","Unnamed: 11"]) # remove empty columns

df.to_csv(os.path.join(order_forms_processed_dir, 'combined_order_forms.csv'), index=False)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 10637 entries, 0 to 1122
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Timestamp                    10637 non-null  object
 1   TEA                          10589 non-null  object
 2   FLAVOR                       4197 non-null   object
 3   MILK                         6873 non-null   object
 4   ICE [Ice Level]              8017 non-null   object
 5   SWEETNESS [Sweetness Level]  8181 non-null   object
 6   Toppings                     9702 non-null   object
 7   NAME                         10628 non-null  object
 8   Started                      10629 non-null  object
 9   Completed                    10623 non-null  object
 10  Form_ID                      10637 non-null  object
dtypes: object(11)
memory usage: 997.2+ KB
None


Since we have a very large sample of orders, I'm just going to remove rows that have incomplete names, started, and complete data since they make up less than 1% of the data and we aren't doing anything super rigorous here.

In [47]:
df = df[(df["NAME"].notna() & df["Started"].notna() & df["Completed"].notna())]

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10606 entries, 0 to 1122
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Timestamp                    10606 non-null  object
 1   TEA                          10559 non-null  object
 2   FLAVOR                       4190 non-null   object
 3   MILK                         6851 non-null   object
 4   ICE [Ice Level]              8000 non-null   object
 5   SWEETNESS [Sweetness Level]  8164 non-null   object
 6   Toppings                     9677 non-null   object
 7   NAME                         10606 non-null  object
 8   Started                      10606 non-null  object
 9   Completed                    10606 non-null  object
 10  Form_ID                      10606 non-null  object
dtypes: object(11)
memory usage: 994.3+ KB


It's looking good! Next, I want to look at the rows where tea data is missing.

In [48]:
blank_tea = df[df["TEA"].isna()]

blank_tea.head(20)

,Timestamp,TEA,FLAVOR,MILK,ICE [Ice Level],SWEETNESS [Sweetness Level],Toppings,NAME,Started,Completed,Form_ID
300,2/16/2023 20:01:42,NaN,NaN,MILK,0%,100%,NaN,Kyle,TRUE,TRUE,2/9/2023 19:06:36
759,2/28/2023 20:14:38,NaN,NaN,NaN,NaN,NaN,BOBA,Stephanie (Fuck it bucket),TRUE,TRUE,2/9/2023 19:06:36
812,2/28/2023 21:17:02,NaN,NaN,NaN,NaN,NaN,NaN,sam fuckit bucket,TRUE,TRUE,2/9/2023 19:06:36
822,2/28/2023 21:25:20,NaN,NaN,NaN,NaN,NaN,NaN,amy fuckit bucket,TRUE,TRUE,2/9/2023 19:06:36
918,3/2/2023 21:37:07,NaN,NaN,Soy,NaN,NaN,BOBA,Raya,TRUE,TRUE,2/9/2023 19:06:36
1120,3/21/2023 20:44:52,NaN,NaN,Soy,NaN,NaN,BOBA,Reia,TRUE,TRUE,2/9/2023 19:06:36
1154,3/21/2023 21:24:09,NaN,NaN,NaN,NaN,NaN,BOBA,1 fuckit Bucket for Ty,TRUE,TRUE,2/9/2023 19:06:36
1220,3/23/2023 20:18:19,NaN,NaN,NaN,NaN,NaN,NaN,shawn - fuck it bucket,TRUE,TRUE,2/9/2023 19:06:36
1268,3/23/2023 21:05:33,NaN,NaN,MILK,50%,100%,BOBA,may,TRUE,TRUE,2/9/2023 19:06:36
1576,4/6/2023 20:00:56,NaN,NaN,NaN,0%,NaN,BOBA,kyle - chai,TRUE,TRUE,2/9/2023 19:06:36


It looks like the tea data is missing for some test runs, really custom orders (I'm looking at you Cindy), or fuckit bucket orders, which I don't want to train our model on. While there's some legitimate orders, I'm going to remove all rows with missing tea data to avoid issues down the road.

In [49]:

df = df[df["TEA"].notna()]

df.to_csv(os.path.join(order_forms_processed_dir, "order_forms_pre_sort.csv"), index=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10559 entries, 0 to 1122
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Timestamp                    10559 non-null  object
 1   TEA                          10559 non-null  object
 2   FLAVOR                       4182 non-null   object
 3   MILK                         6825 non-null   object
 4   ICE [Ice Level]              7980 non-null   object
 5   SWEETNESS [Sweetness Level]  8144 non-null   object
 6   Toppings                     9641 non-null   object
 7   NAME                         10559 non-null  object
 8   Started                      10559 non-null  object
 9   Completed                    10559 non-null  object
 10  Form_ID                      10559 non-null  object
dtypes: object(11)
memory usage: 989.9+ KB


Finally, I want to standardize the timestamps to be in the format of YYYY-MM-DD HH:MM:SS, convert them to datetime objects, and then sort the dataframe by timestamp. Lets check if any timestamps are in the wrong format.

In [50]:
# regex pattern to match example timestamp "2/9/2023 19:11:27"
timestamp_pattern = r'\d{1,2}/\d{1,2}/\d{4}\s\d{2}:\d{2}:\d{2}'

# find rows where timestamp doesn't match the pattern
wrong_format = df[~df["Timestamp"].str.match(timestamp_pattern)]

wrong_format.head()

,Timestamp,TEA,FLAVOR,MILK,ICE [Ice Level],SWEETNESS [Sweetness Level],Toppings,NAME,Started,Completed,Form_ID


Yay! All our timestamps are in the same format (which makes sense since they were all created by google forms). I'm going to convert them to datetime objects, sort the dataframe by timestamp, and save the dataframe to a new csv.

In [51]:
# convert timestamps to datetime and localize to Pacific time
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format='%m/%d/%Y %H:%M:%S').dt.tz_localize('US/Pacific')
df.sort_values(by="Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

df.to_csv(os.path.join(order_forms_processed_dir, "order_forms_sorted.csv")) # saves with reindex